In [ ]:
import requests
import time
import pandas as pd

TOKEN = "d5c3d78fc111d88a0a37b4ab8f83cbd5"
BASE_URL = "https://ows.goszakup.gov.kz"
API_ENDPOINT = "/v3/plans/deleted"

headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}

# Загружаем текущий parquet-файл
parquet_file = "goszakup_deleted_data.parquet"
try:
    df_existing = pd.read_parquet(parquet_file)
    max_index_date = df_existing["index_date"].max()  # Фиксированная максимальная дата
except FileNotFoundError:
    df_existing = pd.DataFrame()
    max_index_date = None

print(f"📌 Найдено {len(df_existing)} записей, max_index_date: {max_index_date}")

next_page = API_ENDPOINT + "?limit=500"
all_data = []
no_new_data_count = 0

while next_page:
    try:
        response = requests.get(BASE_URL + next_page, headers=headers, timeout=30)
        response.raise_for_status()
        data = response.json()

        items = data.get("items", [])
        if not items:
            print("🚫 Данных больше нет, остановка загрузки.")
            break

        first_index_date = items[0].get("index_date", "Нет даты")
        last_index_date = items[-1].get("index_date", "Нет даты")
        print(f"\n🔹 Запрос к API: {BASE_URL + next_page}")
        print(f"📅 Первая index_date: {first_index_date}, Последняя index_date: {last_index_date}")

        # Фильтруем записи с валидной датой
        valid_items = [item for item in items if item.get("index_date") is not None]
        total_items = len(valid_items)

        # Если max_index_date нет, берём все записи
        if max_index_date is None:
            new_items = valid_items
        else:
            # Фильтруем относительно фиксированного max_index_date
            new_items = [item for item in valid_items if item["index_date"] > max_index_date]

        # Диагностика
        print(f"📌 Всего записей на странице: {total_items}")
        print(f"📌 Примеры данных:")
        for i, item in enumerate(valid_items[:3]):
            print(f"  - ID: {item['id']}, index_date: {item.get('index_date', 'Нет даты')}")
        print(f"📌 Новых записей (index_date > {max_index_date}): {len(new_items)}")

        if new_items:
            all_data.extend(new_items)
            print(f"✅ Загружено {len(new_items)} новых записей с этой страницы...")
            no_new_data_count = 0
        else:
            no_new_data_count += 1
            print(f"⚠️ На этой странице нет новых записей ({no_new_data_count}/3).")

        # Если три страницы подряд без новых данных — завершаем
        if no_new_data_count >= 3:
            print("🚫 Три страницы без новых данных, остановка загрузки.")
            break

        # Следующая страница
        next_page = data.get("next_page")
        if not next_page:
            print("🏁 Достигнута последняя страница, завершаем загрузку.")
            break

        time.sleep(5)

    except requests.exceptions.RequestException as e:
        print(f"❌ Ошибка запроса: {e}")
        print("⏳ Ждём 30 сек и пробуем снова...")
        time.sleep(30)

# Сохранение данных
if all_data:
    df_new = pd.DataFrame(all_data)
    df_combined = pd.concat([df_existing, df_new], ignoring_index=True)
    df_combined = df_combined.drop_duplicates(subset=["id"], keep="last")
    df_combined.to_parquet(parquet_file, index=False, engine="pyarrow")
    print(f"🎉 Дозагрузка завершена, сохранено {len(df_combined)} записей в {parquet_file}")
else:
    print("🔍 Новых данных не найдено.")

📌 Найдено 19875273 записей, max_index_date: 2025-03-28 14:10:17

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?limit=500
📅 Первая index_date: 2025-03-31 20:01:24, Последняя index_date: 2025-03-31 19:41:12
📌 Всего записей на странице: 499
📌 Примеры данных:
  - ID: 78409647, index_date: 2025-03-31 20:01:24
  - ID: 78409635, index_date: 2025-03-31 20:01:25
  - ID: 78409628, index_date: 2025-03-31 20:01:25
📌 Новых записей (index_date > 2025-03-28 14:10:17): 499
✅ Загружено 499 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78409096
📅 Первая index_date: 2025-03-31 19:41:12, Последняя index_date: 2025-03-31 19:21:12
📌 Всего записей на странице: 500
📌 Примеры данных:
  - ID: 78409095, index_date: 2025-03-31 19:41:12
  - ID: 78409094, index_date: 2025-03-31 19:41:12
  - ID: 78409092, index_date: 2025-03-31 20:01:25
📌 Новых записей (index_date > 2025-03-28 14:10:17): 500
✅ Загружено 500 новых записей

❌ Ошибка запроса: HTTPSConnectionPool(host='ows.goszakup.gov.kz', port=443): Read timed out. (read timeout=30)
⏳ Ждём 30 сек и пробуем снова...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78394516
📅 Первая index_date: 2025-03-31 17:43:50, Последняя index_date: 2025-03-31 12:11:22
📌 Всего записей на странице: 492
📌 Примеры данных:
  - ID: 78394513, index_date: 2025-03-31 17:43:50
  - ID: 78394510, index_date: 2025-03-31 17:44:05
  - ID: 78394509, index_date: 2025-03-31 19:01:32
📌 Новых записей (index_date > 2025-03-28 14:10:17): 492
✅ Загружено 492 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78393507
📅 Первая index_date: 2025-03-31 14:25:56, Последняя index_date: 2025-03-31 12:10:53
📌 Всего записей на странице: 496
📌 Примеры данных:
  - ID: 78393506, index_date: 2025-03-31 14:25:56
  - ID: 78393505, index_date: 2025-03-31 12:09:45
  - ID: 78393504, index


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78372175
📅 Первая index_date: 2025-03-31 12:06:17, Последняя index_date: 2025-03-31 12:06:58
📌 Всего записей на странице: 473
📌 Примеры данных:
  - ID: 78372172, index_date: 2025-03-31 12:06:17
  - ID: 78372168, index_date: 2025-03-31 12:09:05
  - ID: 78372165, index_date: 2025-03-31 17:44:23
📌 Новых записей (index_date > 2025-03-28 14:10:17): 473
✅ Загружено 473 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78370294
📅 Первая index_date: 2025-03-31 12:12:01, Последняя index_date: 2025-03-31 12:13:17
📌 Всего записей на странице: 481
📌 Примеры данных:
  - ID: 78370289, index_date: 2025-03-31 12:12:01
  - ID: 78370288, index_date: 2025-03-31 12:10:14
  - ID: 78370287, index_date: 2025-03-31 12:09:14
📌 Новых записей (index_date > 2025-03-28 14:10:17): 481
✅ Загружено 481 новых записей с этой страницы...

🔹 Запрос к 


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78332937
📅 Первая index_date: 2025-03-31 12:07:45, Последняя index_date: 2025-03-31 12:13:22
📌 Всего записей на странице: 425
📌 Примеры данных:
  - ID: 78332920, index_date: 2025-03-31 12:07:45
  - ID: 78332918, index_date: 2025-03-31 12:14:17
  - ID: 78332916, index_date: 2025-03-31 12:07:31
📌 Новых записей (index_date > 2025-03-28 14:10:17): 422
✅ Загружено 422 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78330205
📅 Первая index_date: 2025-03-31 17:43:55, Последняя index_date: None
📌 Всего записей на странице: 439
📌 Примеры данных:
  - ID: 78330204, index_date: 2025-03-31 17:43:55
  - ID: 78330201, index_date: 2025-03-31 12:11:50
  - ID: 78330191, index_date: 2025-03-31 12:09:14
📌 Новых записей (index_date > 2025-03-28 14:10:17): 436
✅ Загружено 436 новых записей с этой страницы...

🔹 Запрос к API: https://ow


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78256087
📅 Первая index_date: 2025-03-31 12:09:01, Последняя index_date: 2025-03-31 12:06:35
📌 Всего записей на странице: 377
📌 Примеры данных:
  - ID: 78256086, index_date: 2025-03-31 12:09:01
  - ID: 78256076, index_date: 2025-03-31 12:12:09
  - ID: 78256067, index_date: 2025-03-31 12:07:58
📌 Новых записей (index_date > 2025-03-28 14:10:17): 374
✅ Загружено 374 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=78251588
📅 Первая index_date: 2025-03-31 12:13:03, Последняя index_date: None
📌 Всего записей на странице: 356
📌 Примеры данных:
  - ID: 78251583, index_date: 2025-03-31 12:13:03
  - ID: 78251572, index_date: 2025-03-31 12:10:53
  - ID: 78251568, index_date: 2025-03-31 12:09:01
📌 Новых записей (index_date > 2025-03-28 14:10:17): 351
✅ Загружено 351 новых записей с этой страницы...

🔹 Запрос к API: https://ow

...


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77630575
📅 Первая index_date: 2025-03-17 02:23:50, Последняя index_date: 2025-03-31 12:12:17
📌 Всего записей на странице: 402
📌 Примеры данных:
  - ID: 77630569, index_date: 2025-03-17 02:23:50
  - ID: 77630563, index_date: 2025-02-28 17:41:14
  - ID: 77630549, index_date: 2025-03-31 12:13:53
📌 Новых записей (index_date > 2025-03-28 14:10:17): 348
✅ Загружено 348 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77627906
📅 Первая index_date: 2025-03-31 12:10:24, Последняя index_date: 2025-03-31 12:07:46
📌 Всего записей на странице: 370
📌 Примеры данных:
  - ID: 77627903, index_date: 2025-03-31 12:10:24
  - ID: 77627898, index_date: 2025-03-31 12:09:00
  - ID: 77627895, index_date: 2025-03-31 12:06:23
📌 Новых записей (index_date > 2025-03-28 14:10:17): 309
✅ Загружено 309 новых записей с этой страницы...

🔹 Запрос к 


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77589495
📅 Первая index_date: 2025-03-31 12:11:20, Последняя index_date: 2025-02-28 17:41:14
📌 Всего записей на странице: 378
📌 Примеры данных:
  - ID: 77589489, index_date: 2025-03-31 12:11:20
  - ID: 77589485, index_date: 2025-03-31 12:12:52
  - ID: 77589484, index_date: 2025-02-28 17:41:14
📌 Новых записей (index_date > 2025-03-28 14:10:17): 327
✅ Загружено 327 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77587362
📅 Первая index_date: 2025-03-31 12:11:05, Последняя index_date: 2025-02-28 17:41:14
📌 Всего записей на странице: 416
📌 Примеры данных:
  - ID: 77587360, index_date: 2025-03-31 12:11:05
  - ID: 77587354, index_date: 2025-03-31 12:13:26
  - ID: 77587349, index_date: 2025-03-31 12:12:19
📌 Новых записей (index_date > 2025-03-28 14:10:17): 314
✅ Загружено 314 новых записей с этой страницы...

🔹 Запрос к 


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77550909
📅 Первая index_date: 2025-03-31 12:09:56, Последняя index_date: 2025-03-31 12:12:03
📌 Всего записей на странице: 419
📌 Примеры данных:
  - ID: 77550905, index_date: 2025-03-31 12:09:56
  - ID: 77550896, index_date: 2025-03-31 12:09:29
  - ID: 77550889, index_date: 2025-02-28 17:41:14
📌 Новых записей (index_date > 2025-03-28 14:10:17): 350
✅ Загружено 350 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77548871
📅 Первая index_date: 2025-03-31 12:14:20, Последняя index_date: 2025-02-28 17:41:14
📌 Всего записей на странице: 449
📌 Примеры данных:
  - ID: 77548867, index_date: 2025-03-31 12:14:20
  - ID: 77548863, index_date: 2025-03-31 12:11:14
  - ID: 77548848, index_date: 2025-03-31 12:08:09
📌 Новых записей (index_date > 2025-03-28 14:10:17): 306
✅ Загружено 306 новых записей с этой страницы...

🔹 Запрос к 


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77506835
📅 Первая index_date: 2025-03-31 12:14:14, Последняя index_date: 2025-03-31 12:13:27
📌 Всего записей на странице: 376
📌 Примеры данных:
  - ID: 77506819, index_date: 2025-03-31 12:14:14
  - ID: 77506798, index_date: 2025-03-31 12:53:56
  - ID: 77506796, index_date: 2025-03-31 12:11:25
📌 Новых записей (index_date > 2025-03-28 14:10:17): 296
✅ Загружено 296 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77504362
📅 Первая index_date: 2025-03-31 12:13:18, Последняя index_date: 2025-03-31 12:09:18
📌 Всего записей на странице: 393
📌 Примеры данных:
  - ID: 77504353, index_date: 2025-03-31 12:13:18
  - ID: 77504349, index_date: 2025-03-31 12:13:54
  - ID: 77504348, index_date: 2025-03-31 12:06:49
📌 Новых записей (index_date > 2025-03-28 14:10:17): 320
✅ Загружено 320 новых записей с этой страницы...

🔹 Запрос к 


🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77469201
📅 Первая index_date: 2025-03-31 12:14:22, Последняя index_date: 2025-02-28 17:41:14
📌 Всего записей на странице: 432
📌 Примеры данных:
  - ID: 77469194, index_date: 2025-03-31 12:14:22
  - ID: 77469192, index_date: 2025-03-31 12:11:22
  - ID: 77469191, index_date: 2025-03-31 21:01:07
📌 Новых записей (index_date > 2025-03-28 14:10:17): 384
✅ Загружено 384 новых записей с этой страницы...

🔹 Запрос к API: https://ows.goszakup.gov.kz/v3/plans/deleted?page=next&limit=500&search_after=77466938
📅 Первая index_date: None, Последняя index_date: 2025-02-28 17:41:14
📌 Всего записей на странице: 439
📌 Примеры данных:
  - ID: 77466935, index_date: 2025-03-31 12:06:37
  - ID: 77466934, index_date: 2025-03-31 12:07:30
  - ID: 77466930, index_date: 2025-03-31 12:11:58
📌 Новых записей (index_date > 2025-03-28 14:10:17): 309
✅ Загружено 309 новых записей с этой страницы...

🔹 Запрос к API: https://ow

In [25]:
df_combined = df_combined.sort_values(by='index_date')
df_combined.head()

,id,rootrecord_id,is_deleted,index_date
9937885,3131588,2097933.0,1,2025-02-28 17:41:14
13242428,59721376,59721376.0,1,2025-02-28 17:41:14
13242427,59721378,59721378.0,1,2025-02-28 17:41:14
13242426,59721381,56700930.0,1,2025-02-28 17:41:14
13242425,59721384,58511109.0,1,2025-02-28 17:41:14


In [21]:
df_combined.tail(501)

,id,rootrecord_id,is_deleted,index_date
18772863,77267462,74964094.0,1,2025-03-28 14:10:17
18772843,77267552,74951894.0,1,2025-03-28 14:10:17
19875695,78409180,NaN,1,2025-03-31 19:41:11
19875696,78409179,NaN,1,2025-03-31 19:41:11
19875697,78409178,NaN,1,2025-03-31 19:41:11
...,...,...,...,...
19875564,78409323,NaN,1,2025-03-31 20:01:25
19875563,78409324,NaN,1,2025-03-31 20:01:25
19875562,78409325,NaN,1,2025-03-31 20:01:25
19875568,78409318,NaN,1,2025-03-31 20:01:25
